## __Tentative de fine tuning de mobilenetV2__

In [67]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchmetrics.classification import MulticlassF1Score
from lib.data.preprocessing import TorchPreprocessor
from lib.data.dataset import BeeDataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import torch.optim as optim

import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"L'entraînement se fera sur : {device}")

L'entraînement se fera sur : cuda


In [68]:
# Load model directly from Hugging Face
num_labels = 50
base_processor = AutoImageProcessor.from_pretrained("google/mobilenet_v2_1.0_224")
model = AutoModelForImageClassification.from_pretrained("google/mobilenet_v2_1.0_224", num_labels=num_labels,ignore_mismatched_sizes=True)

Some weights of MobileNetV2ForImageClassification were not initialized from the model checkpoint at google/mobilenet_v2_1.0_224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1001]) in the checkpoint and torch.Size([50]) in the model instantiated
- classifier.weight: found shape torch.Size([1001, 1280]) in the checkpoint and torch.Size([50, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Le preprocessor fourni ne pad pas, ce qui nous fait perdre des abeilles. On se propose donc de remplacer cette partie du code

In [69]:
mobile_net_processor_parameters= {
    "mean" : base_processor.image_mean,
    "std" : base_processor.image_std,
    "crop_size" : (base_processor.crop_size["height"], base_processor.crop_size["width"])
}

preprocessor = TorchPreprocessor(
    normalize=True,
    resize_method="pad",
    target_size=mobile_net_processor_parameters["crop_size"],
)

dataset = BeeDataset(
    root_dir="data/train",
    transform=preprocessor
)

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [70]:
# 1. Configuration (Backbone gelé, Tête active)
for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

# Filtrer les paramètres pour l'optimiseur (plus propre pour la mémoire)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
criterion = torch.nn.CrossEntropyLoss()

# 2. Boucle d'entraînement
model.train()
num_epochs = 30
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

f1_metric = MulticlassF1Score(num_classes=num_labels, average='macro').to(device)
model.to(device)

for epoch in range(num_epochs):
    loop = tqdm(dataloader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True)
    
    epoch_loss = 0
    
    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        logits = outputs.logits
        loss = criterion(logits, labels)
        
        loss.backward()
        optimizer.step()
        
        # --- Calcul du F1 Score ---
        preds = torch.argmax(logits, dim=-1)
        f1_metric.update(preds, labels)
        
        # Affichage dynamique
        current_f1 = f1_metric.compute()
        epoch_loss += loss.item()
        loop.set_postfix(
            loss=f"{loss.item():.4f}", 
            f1=f"{current_f1.item():.4f}"
        )
    # Score final de l'epoch
    final_f1 = f1_metric.compute()
    print(f"-> Fin Epoch {epoch+1} | Loss: {epoch_loss/len(dataloader):.4f} | F1: {final_f1:.4f}")

Epoch [1/30]:   0%|          | 0/250 [00:00<?, ?it/s]

-> Fin Epoch 1 | Loss: 2.3082 | F1: 0.1303


Epoch [2/30]:   0%|          | 0/250 [00:00<?, ?it/s]

-> Fin Epoch 2 | Loss: 1.6595 | F1: 0.2411


Epoch [3/30]:   0%|          | 0/250 [00:00<?, ?it/s]

...

-> Fin Epoch 28 | Loss: 0.9622 | F1: 0.7906


Epoch [29/30]:   0%|          | 0/250 [00:00<?, ?it/s]

-> Fin Epoch 29 | Loss: 0.9762 | F1: 0.7929


Epoch [30/30]:   0%|          | 0/250 [00:00<?, ?it/s]

-> Fin Epoch 30 | Loss: 0.9718 | F1: 0.7950


In [71]:
class JITWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        
    def forward(self, x):
        # On extrait uniquement le Tensor des logits pour satisfaire le JIT
        outputs = self.model(x)
        return outputs.logits

model.eval()
model.to("cpu") # L'export est plus stable sur CPU
wrapper = JITWrapper(model)

dummy_input = torch.randn(1, 3, 224, 224)
traced_model = torch.jit.trace(wrapper, dummy_input)

traced_model.save("models/mobilenet_v2_bees_jit.pt")

/home/alexandre-tonon/anaconda3/envs/pie_env/lib/python3.12/site-packages/transformers/models/mobilenet_v2/modeling_mobilenet_v2.py:230: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  in_height = int(features.shape[-2])
/home/alexandre-tonon/anaconda3/envs/pie_env/lib/python3.12/site-packages/transformers/models/mobilenet_v2/modeling_mobilenet_v2.py:231: TracerWarning: Converting a tensor to a Python integer might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  in_width = int(features.shape[-1])


In [72]:
loaded_jit_model = torch.jit.load("models/mobilenet_v2_bees_jit.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_jit_model.to(device)
loaded_jit_model.eval()

with torch.no_grad():
    # Assure-toi que les images sont aussi sur le même device
    images = images.to(device)
    results = loaded_jit_model(images)

In [73]:
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from PIL import Image

In [74]:
class BeeDataset2(Dataset):
    def __init__(self, train, transform=None):
        self.train_csv_dir = "data/train.csv"
        self.test_csv_dir = "data/test.csv"
        
        self.train = train

        self.transform = transform

        if train:
            file = pd.read_csv(self.train_csv_dir, sep=",")
            image_paths = file["id"]
            label = file["label"]
            image_paths = [os.path.join("data/", img) for img in image_paths]
            self.samples = list(zip(image_paths, label))

        else:
            file = pd.read_csv(self.test_csv_dir, sep=",")
            id = file["id"]
            image_path = file["image"]
            image_paths = [os.path.join("data/test", img) for img in image_path]
            self.samples = list(zip(image_paths, id))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
            path, extra = self.samples[idx] # extra = label (train) ou id (test)

            img = Image.open(path).convert("RGB")
            
            if self.transform:
                img = self.transform(img)

            if self.train:
                return img, torch.tensor(extra, dtype=torch.long)
            else:
                return img, extra

In [75]:
test_dataset = BeeDataset2(train=False, transform=preprocessor)

loaded_jit_model.eval()

all_results = [] 
with torch.no_grad():
    for images, ids in test_dataset:
        images = images.to(device)
        logits = loaded_jit_model(images.unsqueeze(0))
        predicted_label = torch.argmax(logits, dim=-1).item()
        
        all_results.append({"id": ids, "label": predicted_label})

submission_results = pd.DataFrame(all_results)

# Sauvegarde finale
submission_results.to_csv("submission1.csv", index=False)
print(f"✅ Terminé ! {len(submission_results)} prédictions sauvegardées.")

✅ Terminé ! 1997 prédictions sauvegardées.
